# Lab 9: Dynamical Systems, Markov Chains, and Perron--Frobenius Theory

Independent-study notebook with solutions.

This lab studies repeated matrix multiplication
$$
x_{k+1}=Ax_k.
$$

In [ ]:
import numpy as np
from numpy.linalg import eig, solve, matrix_power
np.set_printoptions(precision=4, suppress=True)

## 1. Simulating a discrete dynamical system

Given $x_{k+1}=Ax_k$, compute several iterates.

In [ ]:
A = np.array([[0.6, 0.3, 0.4],
              [0.1, 0.4, 0.3],
              [0.3, 0.3, 0.3]])
x = np.array([1.0, 0.0, 0.0])
for k in range(8):
    print(k, x)
    x = A @ x

**Answer.** The state vector approaches a stable distribution. The matrix is column stochastic, so total mass remains 1.

## 2. Check whether a matrix is column stochastic

In [ ]:
def is_column_stochastic(A, tol=1e-10):
    return np.all(A >= -tol) and np.allclose(A.sum(axis=0), np.ones(A.shape[1]))

is_column_stochastic(A), A.sum(axis=0)

**Similar practice.** Create a $3	imes 3$ column-stochastic matrix and verify it.

In [ ]:
B = np.array([[0.7, 0.2, 0.1],
              [0.2, 0.5, 0.3],
              [0.1, 0.3, 0.6]])
is_column_stochastic(B), B.sum(axis=0)

## 3. Stationary distribution

Solve $A\pi=\pi$ and $\sum_i\pi_i=1$.

In [ ]:
A2 = np.array([[0.9, 0.2],
               [0.1, 0.8]])
M = A2 - np.eye(2)
C = np.vstack([M[0], np.ones(2)])
d = np.array([0.0, 1.0])
pi = solve(C, d)
pi

**Answer.** The stationary distribution is $\pi=(2/3,1/3)^T$.

**Similar practice.** Find the stationary distribution of
$$
A=egin{bmatrix}0.8&0.3\0.2&0.7\end{bmatrix}.
$$

In [ ]:
A_practice = np.array([[0.8, 0.3],
                       [0.2, 0.7]])
M = A_practice - np.eye(2)
C = np.vstack([M[0], np.ones(2)])
solve(C, np.array([0.0, 1.0]))

**Answer.** $(0.6,0.4)^T$.

## 4. Power iteration

In [ ]:
def power_iteration(A, x0=None, steps=30):
    n = A.shape[0]
    x = np.ones(n)/n if x0 is None else x0.astype(float)
    hist = [x.copy()]
    for k in range(steps):
        x = A @ x
        if abs(x.sum()) > 1e-14:
            x = x / x.sum()
        hist.append(x.copy())
    return np.array(hist)

hist = power_iteration(A2, np.array([0.5,0.5]), 15)
hist[-1]

**Answer.** The iterates converge to the stationary distribution.

## 5. Periodic chain

In [ ]:
P = np.array([[0,1],[1,0]], dtype=float)
x = np.array([1.0,0.0])
for k in range(6):
    print(k, x)
    x = P @ x

**Answer.** The sequence alternates. The chain has a stationary distribution $(1/2,1/2)^T$, but not every initial state converges to it.

## 6. PageRank damping

In [ ]:
P = np.array([[0,   1/2, 1],
              [1/2, 0,   0],
              [1/2, 1/2, 0]])
alpha = 0.85
n = 3
G = alpha*P + (1-alpha)/n*np.ones((n,n))
print(G)
print(G.sum(axis=0))
hist = power_iteration(G, steps=100)
hist[-1]

**Answer.** The damping matrix is positive, hence primitive. The limiting vector is the PageRank vector.

## 7. Perron eigenvector for a positive matrix

In [ ]:
C = np.array([[2,1,1],
              [1,3,1],
              [1,1,2]], dtype=float)
w, V = eig(C)
idx = np.argmax(w.real)
rho = w[idx].real
v = V[:, idx].real
v = np.abs(v)
v = v / v.sum()
rho, v

**Answer.** The dominant eigenvalue is the Perron eigenvalue. The normalized positive eigenvector gives the stable positive direction.

## 8. Leslie population model

In [ ]:
L = np.array([[0.2, 1.4],
              [0.6, 0.0]])
w, V = eig(L)
idx = np.argmax(np.abs(w))
print('dominant eigenvalue:', w[idx].real)
stable = V[:,idx].real
stable = np.abs(stable)/np.abs(stable).sum()
print('stable stage distribution:', stable)

x = np.array([100.0, 40.0])
for k in range(10):
    print(k, x, 'normalized=', x/x.sum())
    x = L @ x

**Answer.** The dominant eigenvalue is the long-term growth factor. The normalized population vector approaches the stable stage distribution.

## 9. Reflection

Write a short response:

1. Which example best explains stationary distributions?
2. What is the difference between convergence and having a stationary distribution?
3. How does Perron--Frobenius theory explain the success of PageRank?